# 🫀 Demo Cardiac Segmentation — UNet URPC (Boundary-Aware)

**Author:** KhangPX Improvement — Demo by AI Assistant  
**Model:** `UNet_URPC` (Uncertainty Rectified Pyramid Consistency)  
**Dataset:** ACDC — 4 classes: Background, RV (Right Ventricle), Myo (Myocardium), LV (Left Ventricle)  
**Input:** Grayscale cardiac image (any format: `.png`, `.jpg`, `.nii.gz`, `.h5`)  
**Output:** Segmentation mask with color overlay

---

## 1. Install & Import Dependencies

In [ ]:
# Uncomment if running on Colab/Kaggle and packages are missing:
# !pip install torch torchvision matplotlib numpy Pillow scipy
# !pip install nibabel h5py SimpleITK medpy  # for NIfTI / HDF5 support

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy.ndimage import zoom as scipy_zoom

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:     {torch.cuda.get_device_name(0)}")

## 2. Define UNet-URPC Model Architecture
The model is defined inline so this notebook is **self-contained** — no need to import from
the project's `networks/` directory.

In [ ]:
from torch.distributions.uniform import Uniform

# ─── Building Blocks ────────────────────────────────────────────

class ConvBlock(nn.Module):
    """Two conv layers + BN + LeakyReLU + optional Dropout."""
    def __init__(self, in_channels, out_channels, dropout_p):
        super().__init__()
        self.conv_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(),
            nn.Dropout(dropout_p),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(),
        )
    def forward(self, x):
        return self.conv_conv(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        super().__init__()
        self.maxpool_conv = nn.Sequential(nn.MaxPool2d(2), ConvBlock(in_ch, out_ch, dropout_p))
    def forward(self, x):
        return self.maxpool_conv(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch1, in_ch2, out_ch, dropout_p, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        if bilinear:
            self.conv1x1 = nn.Conv2d(in_ch1, in_ch2, 1)
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_ch1, in_ch2, 2, stride=2)
        self.conv = ConvBlock(in_ch2 * 2, out_ch, dropout_p)

    def forward(self, x1, x2):
        if self.bilinear:
            x1 = self.conv1x1(x1)
        x1 = self.up(x1)
        return self.conv(torch.cat([x2, x1], dim=1))


# ─── Encoder ─────────────────────────────────────────────────────

class Encoder(nn.Module):
    def __init__(self, params):
        super().__init__()
        ft = params['feature_chns']
        dp = params['dropout']
        self.in_conv = ConvBlock(params['in_chns'], ft[0], dp[0])
        self.down1 = DownBlock(ft[0], ft[1], dp[1])
        self.down2 = DownBlock(ft[1], ft[2], dp[2])
        self.down3 = DownBlock(ft[2], ft[3], dp[3])
        self.down4 = DownBlock(ft[3], ft[4], dp[4])

    def forward(self, x):
        x0 = self.in_conv(x)
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        x3 = self.down3(x2)
        x4 = self.down4(x3)
        return [x0, x1, x2, x3, x4]


# ─── Helper perturbation functions (used only during training) ──

def Dropout(x, p=0.3):
    return F.dropout(x, p)

def FeatureDropout(x):
    attn = torch.mean(x, dim=1, keepdim=True)
    max_val, _ = torch.max(attn.view(x.size(0), -1), dim=1, keepdim=True)
    threshold = max_val * np.random.uniform(0.7, 0.9)
    threshold = threshold.view(x.size(0), 1, 1, 1).expand_as(attn)
    return x.mul((attn < threshold).float())

class FeatureNoise(nn.Module):
    def __init__(self, uniform_range=0.3):
        super().__init__()
        self.uni_dist = Uniform(-uniform_range, uniform_range)
    def forward(self, x):
        noise = self.uni_dist.sample(x.shape[1:]).to(x.device).unsqueeze(0)
        return x.mul(noise) + x


# ─── Decoder URPC ───────────────────────────────────────────────

class Decoder_URPC(nn.Module):
    def __init__(self, params):
        super().__init__()
        ft = params['feature_chns']
        n_class = params['class_num']
        self.up1 = UpBlock(ft[4], ft[3], ft[3], 0.0)
        self.up2 = UpBlock(ft[3], ft[2], ft[2], 0.0)
        self.up3 = UpBlock(ft[2], ft[1], ft[1], 0.0)
        self.up4 = UpBlock(ft[1], ft[0], ft[0], 0.0)
        self.out_conv     = nn.Conv2d(ft[0], n_class, 3, padding=1)
        self.out_conv_dp4 = nn.Conv2d(ft[4], n_class, 3, padding=1)
        self.out_conv_dp3 = nn.Conv2d(ft[3], n_class, 3, padding=1)
        self.out_conv_dp2 = nn.Conv2d(ft[2], n_class, 3, padding=1)
        self.out_conv_dp1 = nn.Conv2d(ft[1], n_class, 3, padding=1)
        self.feature_noise = FeatureNoise()

    def forward(self, feature, shape):
        x0, x1, x2, x3, x4 = feature
        x = self.up1(x4, x3)
        dp3 = self.out_conv_dp3(x) if not self.training else self.out_conv_dp3(Dropout(x, 0.5))
        dp3 = F.interpolate(dp3, shape)
        x = self.up2(x, x2)
        dp2 = self.out_conv_dp2(x) if not self.training else self.out_conv_dp2(FeatureDropout(x))
        dp2 = F.interpolate(dp2, shape)
        x = self.up3(x, x1)
        dp1 = self.out_conv_dp1(x) if not self.training else self.out_conv_dp1(self.feature_noise(x))
        dp1 = F.interpolate(dp1, shape)
        x = self.up4(x, x0)
        dp0 = self.out_conv(x)
        return dp0, dp1, dp2, dp3


# ─── Full UNet_URPC ─────────────────────────────────────────────

class UNet_URPC(nn.Module):
    def __init__(self, in_chns=1, class_num=4):
        super().__init__()
        params = {
            'in_chns': in_chns,
            'feature_chns': [16, 32, 64, 128, 256],
            'dropout': [0.05, 0.1, 0.2, 0.3, 0.5],
            'class_num': class_num,
            'bilinear': False,
            'acti_func': 'relu',
        }
        self.encoder = Encoder(params)
        self.decoder = Decoder_URPC(params)

    def forward(self, x):
        shape = x.shape[2:]
        feature = self.encoder(x)
        return self.decoder(feature, shape)


print("✅  Model architecture defined successfully.")

## 3. Load Pre-trained Weights

In [ ]:
# ── Configuration ──────────────────────────────────────────────
NUM_CLASSES = 4          # Background, RV, Myo, LV
IN_CHANNELS = 1          # Grayscale
INPUT_SIZE  = 256        # Model was trained with 256×256 patches

# Path to the weight file (adjust if needed)
WEIGHT_PATH = "unet_urpc_best_model (1).pth"

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Build model & load weights ─────────────────────────────────
model = UNet_URPC(in_chns=IN_CHANNELS, class_num=NUM_CLASSES)
model = model.to(device)

checkpoint = torch.load(WEIGHT_PATH, map_location=device, weights_only=False)

# Handle both checkpoint formats: raw state_dict or dict with 'model_state_dict'
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint  iter={checkpoint.get('iteration','N/A')}  dice={checkpoint.get('best_performance','N/A')}")
else:
    model.load_state_dict(checkpoint)

model.eval()
print(f"✅  Model loaded successfully from '{WEIGHT_PATH}'")
print(f"    Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Utility Functions

In [ ]:
# ── Class labels & color map ───────────────────────────────────
CLASS_NAMES  = ["Background", "RV (Right Ventricle)", "Myo (Myocardium)", "LV (Left Ventricle)"]
CLASS_COLORS = [
    [0,   0,   0],     # Background — black (transparent)
    [255, 0,   0],     # RV  — red
    [0,   255, 0],     # Myo — green
    [0,   0,   255],   # LV  — blue
]


def load_image(image_path):
    """
    Load a medical / standard image and return a 2D numpy array (H, W).
    Supports: .png, .jpg, .bmp, .tif, .nii, .nii.gz, .h5
    For 3D volumes (.nii.gz, .h5), returns the middle slice.
    """
    ext = os.path.splitext(image_path)[-1].lower()
    if image_path.endswith('.nii.gz'):
        ext = '.nii.gz'

    if ext in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'):
        img = Image.open(image_path).convert('L')  # → grayscale
        return np.array(img, dtype=np.float32)

    elif ext in ('.nii', '.nii.gz'):
        import nibabel as nib
        vol = nib.load(image_path).get_fdata().astype(np.float32)
        # Take middle slice along the last (z) axis
        mid = vol.shape[-1] // 2
        return vol[:, :, mid]

    elif ext == '.h5':
        import h5py
        with h5py.File(image_path, 'r') as f:
            vol = f['image'][:]
        # vol is (D, H, W) — take middle slice
        mid = vol.shape[0] // 2
        return vol[mid].astype(np.float32)

    else:
        raise ValueError(f"Unsupported image format: {ext}")


def load_label(image_path):
    """
    Load the ground truth label corresponding to the image.
    Supports: .h5 (key='label'), .nii.gz (looks for matching _gt.nii.gz file)
    Returns a 2D integer numpy array (H, W) or None if no GT is available.
    """
    ext = os.path.splitext(image_path)[-1].lower()
    if image_path.endswith('.nii.gz'):
        ext = '.nii.gz'

    if ext == '.h5':
        import h5py
        with h5py.File(image_path, 'r') as f:
            if 'label' in f:
                vol = f['label'][:]
                mid = vol.shape[0] // 2
                return vol[mid].astype(np.int32)
            else:
                print("⚠️  No 'label' key found in .h5 file.")
                return None

    elif ext in ('.nii', '.nii.gz'):
        # Try to find a matching ground truth file
        # Common patterns: *_gt.nii.gz, *_seg.nii.gz
        import nibabel as nib
        base = image_path.replace('.nii.gz', '').replace('.nii', '')
        for suffix in ['_gt.nii.gz', '_seg.nii.gz', '_label.nii.gz']:
            gt_path = base + suffix
            if os.path.exists(gt_path):
                vol = nib.load(gt_path).get_fdata().astype(np.int32)
                mid = vol.shape[-1] // 2
                return vol[:, :, mid]
        print("⚠️  No ground truth NIfTI file found (tried _gt, _seg, _label suffixes).")
        return None

    else:
        # For standard images (.png, .jpg, etc.) there's no built-in GT
        print("ℹ️  Ground truth not available for standard image formats.")
        return None


def preprocess(img_2d, target_size=256):
    """
    Resize to (target_size, target_size) and normalize to [0, 1].
    Returns a torch Tensor of shape (1, 1, H, W).
    """
    h, w = img_2d.shape
    resized = scipy_zoom(img_2d, (target_size / h, target_size / w), order=1)
    # Normalize to [0, 1]
    vmin, vmax = resized.min(), resized.max()
    if vmax - vmin > 1e-8:
        resized = (resized - vmin) / (vmax - vmin)
    tensor = torch.from_numpy(resized).unsqueeze(0).unsqueeze(0).float()
    return tensor


def predict(model, img_tensor, device):
    """
    Run inference and return the predicted segmentation mask (H, W).
    """
    img_tensor = img_tensor.to(device)
    with torch.no_grad():
        out_main, _, _, _ = model(img_tensor)
    pred = torch.argmax(torch.softmax(out_main, dim=1), dim=1).squeeze(0)
    return pred.cpu().numpy()


def colorize_mask(mask, colors):
    """Convert integer mask (H, W) to RGB image (H, W, 3)."""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id, color in enumerate(colors):
        rgb[mask == cls_id] = color
    return rgb


def visualize(original_2d, pred_mask, gt_label=None, alpha=0.5):
    """
    Display results. If gt_label is provided, shows 4 panels:
      (1) Original image, (2) Ground Truth, (3) Predicted mask, (4) Overlay.
    Otherwise shows 3 panels:
      (1) Original image, (2) Predicted mask, (3) Overlay.
    """
    color_mask = colorize_mask(pred_mask, CLASS_COLORS)

    # Resize mask back to original image size for overlay
    h, w = original_2d.shape
    pred_resized = scipy_zoom(pred_mask.astype(float), (h / pred_mask.shape[0], w / pred_mask.shape[1]), order=0).astype(int)
    color_mask_resized = colorize_mask(pred_resized, CLASS_COLORS)

    # Normalize original for display
    disp = original_2d.copy()
    vmin, vmax = disp.min(), disp.max()
    if vmax - vmin > 1e-8:
        disp = (disp - vmin) / (vmax - vmin)
    disp_rgb = np.stack([disp]*3, axis=-1)  # (H, W, 3)

    # Overlay
    mask_float = color_mask_resized.astype(float) / 255.0
    has_label = (pred_resized > 0)[..., None].astype(float)
    overlay = disp_rgb * (1 - alpha * has_label) + mask_float * alpha * has_label
    overlay = np.clip(overlay, 0, 1)

    # ── Determine number of panels ──────────────────────────────
    has_gt = gt_label is not None
    n_panels = 4 if has_gt else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    # Panel 1: Original Image
    axes[0].imshow(disp, cmap='gray')
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    if has_gt:
        # Resize GT label to original image size
        gt_resized = scipy_zoom(gt_label.astype(float), (h / gt_label.shape[0], w / gt_label.shape[1]), order=0).astype(int)
        color_gt = colorize_mask(gt_resized, CLASS_COLORS)

        # Panel 2: Ground Truth
        axes[1].imshow(color_gt)
        axes[1].set_title('Ground Truth', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Panel 3: Segmentation Mask
        axes[2].imshow(color_mask_resized)
        axes[2].set_title('Segmentation Mask', fontsize=14, fontweight='bold')
        axes[2].axis('off')

        # Panel 4: Overlay
        axes[3].imshow(overlay)
        axes[3].set_title('Overlay', fontsize=14, fontweight='bold')
        axes[3].axis('off')
    else:
        # Panel 2: Segmentation Mask
        axes[1].imshow(color_mask_resized)
        axes[1].set_title('Segmentation Mask', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Panel 3: Overlay
        axes[2].imshow(overlay)
        axes[2].set_title('Overlay', fontsize=14, fontweight='bold')
        axes[2].axis('off')

    # Legend
    patches = [
        mpatches.Patch(color=np.array(c)/255., label=n)
        for n, c in zip(CLASS_NAMES[1:], CLASS_COLORS[1:])
    ]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=12,
               frameon=True, edgecolor='gray')
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()


print("✅  Utility functions ready.")

## 5. Run Segmentation on Your Image

**Thay đổi `IMAGE_PATH` bên dưới** thành đường dẫn tới ảnh y tế về tim của bạn.  
Hỗ trợ các định dạng: `.png`, `.jpg`, `.bmp`, `.tif`, `.nii`, `.nii.gz`, `.h5`

Đối với file 3D (NIfTI, HDF5), notebook sẽ tự động lấy slice giữa.

**Ground truth label** sẽ được tự động load nếu có (file `.h5` chứa key `label`, hoặc file NIfTI có file `_gt.nii.gz` tương ứng).

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 👇 THAY ĐỔI ĐƯỜNG DẪN ẢNH Ở ĐÂY
# ══════════════════════════════════════════════════════════════════
IMAGE_PATH = "your_cardiac_image.png"   # ← Đổi thành file ảnh của bạn
# ══════════════════════════════════════════════════════════════════

# Load image & ground truth label
original = load_image(IMAGE_PATH)
gt_label = load_label(IMAGE_PATH)

print(f"Image shape: {original.shape}  |  dtype: {original.dtype}")
print(f"Value range: [{original.min():.2f}, {original.max():.2f}]")
if gt_label is not None:
    print(f"Ground truth shape: {gt_label.shape}  |  classes: {np.unique(gt_label).tolist()}")
else:
    print("Ground truth: not available")

img_tensor = preprocess(original, target_size=INPUT_SIZE)
pred_mask  = predict(model, img_tensor, device)

# Show unique predicted classes
unique_classes = np.unique(pred_mask)
print(f"\nPredicted classes: {[CLASS_NAMES[c] for c in unique_classes]}")

# Visualize (with GT if available)
visualize(original, pred_mask, gt_label=gt_label, alpha=0.55)

## 6. (Optional) Batch Processing — Multiple Slices from a Volume

Nếu bạn có file `.nii.gz` hoặc `.h5`, cell dưới đây sẽ chạy segmentation trên **toàn bộ slices** và hiển thị lưới kết quả.

In [ ]:
def segment_volume(volume_path, model, device, target_size=256):
    """
    Segment all slices in a 3D volume (.nii.gz or .h5).
    Returns: images (list of 2D arrays), predictions (list of 2D int arrays)
    """
    ext = os.path.splitext(volume_path)[-1].lower()
    if volume_path.endswith('.nii.gz'):
        ext = '.nii.gz'

    if ext in ('.nii', '.nii.gz'):
        import nibabel as nib
        vol = nib.load(volume_path).get_fdata().astype(np.float32)
        # (H, W, D) → iterate over D
        slices = [vol[:, :, i] for i in range(vol.shape[-1])]
    elif ext == '.h5':
        import h5py
        with h5py.File(volume_path, 'r') as f:
            vol = f['image'][:].astype(np.float32)
        # (D, H, W)
        slices = [vol[i] for i in range(vol.shape[0])]
    else:
        raise ValueError("Volume processing only supports .nii.gz and .h5")

    predictions = []
    for s in slices:
        t = preprocess(s, target_size)
        p = predict(model, t, device)
        predictions.append(p)

    return slices, predictions


def show_volume_grid(slices, predictions, max_cols=6, max_show=18):
    """Display a grid of slices with overlaid predictions."""
    n = min(len(slices), max_show)
    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for i in range(n):
        s = slices[i]
        p = predictions[i]
        h, w = s.shape
        p_resized = scipy_zoom(p.astype(float), (h/p.shape[0], w/p.shape[1]), order=0).astype(int)
        color_mask = colorize_mask(p_resized, CLASS_COLORS)

        disp = s.copy()
        vmin, vmax = disp.min(), disp.max()
        if vmax - vmin > 1e-8:
            disp = (disp - vmin) / (vmax - vmin)
        disp_rgb = np.stack([disp]*3, axis=-1)

        has_label = (p_resized > 0)[..., None].astype(float)
        overlay = disp_rgb * (1 - 0.5*has_label) + (color_mask/255.) * 0.5 * has_label
        overlay = np.clip(overlay, 0, 1)

        axes[i].imshow(overlay)
        axes[i].set_title(f"Slice {i}", fontsize=10)
        axes[i].axis('off')

    # Hide unused
    for j in range(n, len(axes)):
        axes[j].axis('off')

    patches = [mpatches.Patch(color=np.array(c)/255., label=n)
               for n, c in zip(CLASS_NAMES[1:], CLASS_COLORS[1:])]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11, frameon=True)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


print("✅  Volume utilities ready.  Uncomment the cell below to run.")

In [ ]:
# ── Uncomment & thay đường dẫn file volume ──────────────────────
# VOLUME_PATH = "path/to/cardiac_volume.nii.gz"  # hoặc .h5
# slices, preds = segment_volume(VOLUME_PATH, model, device)
# print(f"Volume has {len(slices)} slices")
# show_volume_grid(slices, preds, max_cols=6, max_show=18)

## 7. (Optional) Upload Image Interactively (Google Colab)

Nếu chạy trên **Google Colab**, bạn có thể upload ảnh trực tiếp:

In [ ]:
# Only works in Google Colab
try:
    from google.colab import files
    print("Running on Colab! Upload your cardiac image below:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        print(f"\n📄 Processing: {fname}")
        original = load_image(fname)
        gt_label = load_label(fname)
        img_tensor = preprocess(original, target_size=INPUT_SIZE)
        pred_mask = predict(model, img_tensor, device)
        unique_classes = np.unique(pred_mask)
        print(f"   Predicted classes: {[CLASS_NAMES[c] for c in unique_classes]}")
        visualize(original, pred_mask, gt_label=gt_label, alpha=0.55)
except ImportError:
    print("ℹ️  Not running on Colab. Use IMAGE_PATH in Section 5 instead.")

---
## Notes
- **Model:** UNet-URPC with Boundary-Aware Loss (PP4 improvement by KhangPX)
- **Classes:** 4 classes from ACDC dataset
  - `0`: Background
  - `1`: RV (Right Ventricle) — 🔴 Red
  - `2`: Myo (Myocardium) — 🟢 Green  
  - `3`: LV (Left Ventricle) — 🔵 Blue
- **Input:** Single-channel (grayscale) 256×256 image
- Model outputs 4 decoder heads; only the **main head** (`out_main`) is used for inference.